## Theory: Sample Mean Statistics in Feature Space and RKHS

Let $x_1, \ldots, x_N$ be observations with $x_i \in \mathbb{R}^d$, and define the data matrix $X \in \mathbb{R}^{N \times d}$.

### 1) Feature Space (Original Coordinates)
Using the empirical sample mean:

- Mean vector: $\bar{x} = \frac{1}{N}\sum_{i=1}^N x_i$.
- Centered matrix: $X_c = X - \mathbf{1}\bar{x}^\top$.
- Covariance matrix: $\Sigma = \frac{1}{N-1}X_c^\top X_c$.
- Variance matrix (diagonal): $V = \operatorname{diag}(\Sigma_{11}, \ldots, \Sigma_{dd})$.
- Pearson correlation index matrix: $R_{ij} = \frac{\Sigma_{ij}}{\sqrt{\Sigma_{ii}\Sigma_{jj}}}$.

If a feature has zero variance, the code sets the corresponding undefined correlation terms to 0 for numerical stability.

### 2) RKHS (Kernel Representation)
Let $\phi(\cdot)$ be a feature map into RKHS $\mathcal{H}$, with kernel

$$
k(x_i, x_j) = \langle \phi(x_i), \phi(x_j) \rangle_{\mathcal{H}}.
$$

Define the Gram matrix $K \in \mathbb{R}^{N \times N}$, $K_{ij}=k(x_i,x_j)$, and the centering matrix

$$
H = I_N - \frac{1}{N}\mathbf{1}\mathbf{1}^\top, \qquad K_c = H K H.
$$

The RKHS sample mean is represented by coefficients $\alpha_i=1/N$, i.e.,

$$
\mu_\phi = \frac{1}{N}\sum_{i=1}^N \phi(x_i).
$$

In sample coordinates, the code uses:

- RKHS covariance matrix: $C_\phi = \frac{1}{N-1}K_c$.
- RKHS variance matrix: diagonal of $C_\phi$.
- RKHS Pearson matrix: same normalization rule as above.

This avoids explicit construction of $\phi(x)$ while preserving second-order structure induced by the kernel.


In [4]:
# Import MNIST dataset loader
from keras.datasets import mnist
import numpy as np

# Load MNIST and compute the total number of observations N
(X_train, y_train), (X_test, y_test) = mnist.load_data()
N = X_train.shape[0] + X_test.shape[0]

# Example observations vector x in R^N
x = np.random.randn(N).astype(np.float32)
x.shape


(70000,)

In [5]:
from typing import Tuple

def build_feature_matrix_from_mnist(
    X_train: np.ndarray,  # Training images [N_train, 28, 28]
    X_test: np.ndarray,   # Test images [N_test, 28, 28]
    n_samples: int = 600,  # Number of observations N [samples]
) -> np.ndarray:  # Matrix X in R^{N x d} with d = 784
    """Builds a normalized MNIST design matrix in feature space.

    Purpose:
        Assemble X in R^{N x d} from MNIST, where rows are observations and
        columns are pixel features. Normalization to [0, 1] improves numeric
        stability for covariance and kernel computations.

    Parameters:
        X_train: Training MNIST tensor with uint8 pixel values.
        X_test: Test MNIST tensor with uint8 pixel values.
        n_samples: Number of observations N used in the empirical analysis.

    Returns:
        A float64 matrix X with shape [N, 784].
    """
    if n_samples < 2:
        raise ValueError("n_samples must be at least 2 to compute sample covariance.")

    # Stack train/test sets, then build a deterministic subset of size N
    X_all = np.concatenate((X_train, X_test), axis=0)
    if n_samples > X_all.shape[0]:
        raise ValueError(
            f"n_samples={n_samples} is larger than total MNIST observations {X_all.shape[0]}."
        )

    # Flatten 28x28 images to vectors in R^784 and normalize to [0, 1]
    X = X_all[:n_samples].reshape(n_samples, -1).astype(np.float64) / 255.0
    return X


def compute_feature_space_statistics(
    X: np.ndarray,  # Observation matrix X with shape [N, d]
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """Computes sample-mean variance/covariance/correlation in feature space.

    Purpose:
        Use the sample mean to center X, then compute the sample covariance
        matrix in the original feature space, its diagonal variance matrix,
        and the Pearson correlation index matrix.

    Parameters:
        X: Data matrix with N observations and d features.

    Returns:
        mean_vector: Sample mean vector in R^d.
        variance_matrix: Diagonal matrix with feature-wise variances.
        covariance_matrix: Sample covariance matrix in R^{d x d}.
        pearson_corr_matrix: Pearson correlation index matrix in R^{d x d}.
    """
    n_samples = X.shape[0]
    if n_samples < 2:
        raise ValueError("At least two samples are required for sample covariance.")

    # Center observations using the empirical sample mean
    mean_vector = np.mean(X, axis=0)
    X_centered = X - mean_vector

    # Compute covariance in feature space: Sigma = X_c^T X_c / (N - 1)
    covariance_matrix = (X_centered.T @ X_centered) / (n_samples - 1)  # Unbiased estimator

    # Build variance matrix and normalized Pearson correlation matrix
    variance_vector = np.clip(np.diag(covariance_matrix), a_min=0.0, a_max=None)
    variance_matrix = np.diag(variance_vector)
    std_outer = np.sqrt(np.outer(variance_vector, variance_vector))
    pearson_corr_matrix = np.divide(
        covariance_matrix,
        std_outer,
        out=np.zeros_like(covariance_matrix),
        where=std_outer > 0,
    )

    return mean_vector, variance_matrix, covariance_matrix, pearson_corr_matrix


def compute_rbf_gram_matrix(
    X: np.ndarray,  # Observation matrix X with shape [N, d]
    gamma: float,   # RBF bandwidth coefficient
) -> np.ndarray:    # Gram matrix K with shape [N, N]
    """Computes the RBF kernel Gram matrix K_ij = exp(-gamma ||x_i - x_j||^2).

    Purpose:
        Build the kernel matrix needed to represent RKHS moments without
        explicitly constructing feature maps phi(x).

    Parameters:
        X: Data matrix with N observations and d features.
        gamma: Positive RBF parameter. Typical choice: 1 / d.

    Returns:
        Symmetric Gram matrix K in R^{N x N}.
    """
    if gamma <= 0.0:
        raise ValueError("gamma must be strictly positive for the RBF kernel.")

    # Compute all squared pairwise distances with a vectorized identity
    squared_norms = np.sum(X * X, axis=1, keepdims=True)
    squared_distances = squared_norms + squared_norms.T - 2.0 * (X @ X.T)
    squared_distances = np.maximum(squared_distances, 0.0)

    return np.exp(-gamma * squared_distances)


def center_kernel_matrix(
    K: np.ndarray,  # Uncentered Gram matrix [N, N]
) -> np.ndarray:    # Centered Gram matrix K_c = H K H
    """Centers a Gram matrix using the sample-mean centering operator.

    Purpose:
        Apply H = I - (1/N)11^T to obtain centered RKHS inner products.

    Parameters:
        K: Gram matrix of kernel inner products.

    Returns:
        Centered Gram matrix K_c.
    """
    if K.ndim != 2 or K.shape[0] != K.shape[1]:
        raise ValueError("K must be a square 2D matrix.")

    n_samples = K.shape[0]
    H = np.eye(n_samples, dtype=K.dtype) - np.ones((n_samples, n_samples), dtype=K.dtype) / n_samples
    return H @ K @ H


def compute_rkhs_statistics_from_kernel(
    K: np.ndarray,  # Gram matrix K_ij = <phi(x_i), phi(x_j)>
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """Computes sample-mean variance/covariance/correlation in RKHS sample space.

    Purpose:
        Use kernel centering to compute empirical second-order statistics in
        RKHS without explicit feature maps. The covariance matrix is expressed
        in sample coordinates as C_phi = K_c / (N - 1).

    Parameters:
        K: Uncentered Gram matrix in R^{N x N}.

    Returns:
        mean_coefficients: Coefficients alpha for mu_phi = sum_i alpha_i phi(x_i).
        variance_matrix: Diagonal matrix of RKHS sample-space variances.
        covariance_matrix: RKHS sample-space covariance matrix C_phi.
        pearson_corr_matrix: RKHS Pearson correlation index matrix.
    """
    if K.ndim != 2 or K.shape[0] != K.shape[1]:
        raise ValueError("K must be a square 2D matrix.")

    n_samples = K.shape[0]
    if n_samples < 2:
        raise ValueError("At least two samples are required for sample covariance.")

    # Represent the sample mean in RKHS as a linear combination of mapped data
    mean_coefficients = np.full(n_samples, 1.0 / n_samples, dtype=K.dtype)

    # Center Gram matrix and compute covariance in sample coordinates
    K_centered = center_kernel_matrix(K)
    covariance_matrix = K_centered / (n_samples - 1)

    # Build variance matrix and normalized Pearson correlation matrix
    variance_vector = np.clip(np.diag(covariance_matrix), a_min=0.0, a_max=None)
    variance_matrix = np.diag(variance_vector)
    std_outer = np.sqrt(np.outer(variance_vector, variance_vector))
    pearson_corr_matrix = np.divide(
        covariance_matrix,
        std_outer,
        out=np.zeros_like(covariance_matrix),
        where=std_outer > 0,
    )

    return mean_coefficients, variance_matrix, covariance_matrix, pearson_corr_matrix


# Build a deterministic MNIST subset to keep RKHS matrices computationally tractable
N_subset = 600
X_mnist = build_feature_matrix_from_mnist(X_train, X_test, n_samples=N_subset)

# Feature-space statistics via sample mean
mean_x, var_mat_x, cov_mat_x, corr_mat_x = compute_feature_space_statistics(X_mnist)

# RKHS statistics via sample mean and centered RBF kernel
gamma_rbf = 1.0 / X_mnist.shape[1]
K_rbf = compute_rbf_gram_matrix(X_mnist, gamma=gamma_rbf)
mean_phi_coeffs, var_mat_rkhs, cov_mat_rkhs, corr_mat_rkhs = compute_rkhs_statistics_from_kernel(K_rbf)

{
    "N_subset": N_subset,
    "feature_mean_shape": mean_x.shape,
    "feature_variance_shape": var_mat_x.shape,
    "feature_covariance_shape": cov_mat_x.shape,
    "feature_pearson_shape": corr_mat_x.shape,
    "rkhs_mean_coeff_shape": mean_phi_coeffs.shape,
    "rkhs_variance_shape": var_mat_rkhs.shape,
    "rkhs_covariance_shape": cov_mat_rkhs.shape,
    "rkhs_pearson_shape": corr_mat_rkhs.shape,
}


{'N_subset': 600,
 'feature_mean_shape': (784,),
 'feature_variance_shape': (784, 784),
 'feature_covariance_shape': (784, 784),
 'feature_pearson_shape': (784, 784),
 'rkhs_mean_coeff_shape': (600,),
 'rkhs_variance_shape': (600, 600),
 'rkhs_covariance_shape': (600, 600),
 'rkhs_pearson_shape': (600, 600)}

## Analysis of Final Outputs

The next cell computes compact diagnostics from the final matrices:

- matrix shapes (feature space vs. RKHS),
- total variance (`trace` of covariance),
- mean diagonal correlation,
- mean absolute off-diagonal correlation,
- kernel-centering residual (should be close to 0).


In [ ]:
# Build compact diagnostics from the previously computed final outputs
feature_var_diag = np.diag(var_mat_x)
feature_corr_diag = np.diag(corr_mat_x)
feature_offdiag_mask = ~np.eye(corr_mat_x.shape[0], dtype=bool)

rkhs_var_diag = np.diag(var_mat_rkhs)
rkhs_corr_diag = np.diag(corr_mat_rkhs)
rkhs_offdiag_mask = ~np.eye(corr_mat_rkhs.shape[0], dtype=bool)

# Reuse kernel centering to verify numerical centering quality
K_centered = center_kernel_matrix(K_rbf)

analysis_summary = {
    'N_subset': int(N_subset),
    'd_features': int(X_mnist.shape[1]),
    'feature_variance_shape': var_mat_x.shape,
    'feature_covariance_shape': cov_mat_x.shape,
    'feature_pearson_shape': corr_mat_x.shape,
    'rkhs_variance_shape': var_mat_rkhs.shape,
    'rkhs_covariance_shape': cov_mat_rkhs.shape,
    'rkhs_pearson_shape': corr_mat_rkhs.shape,
    'feature_cov_trace': float(np.trace(cov_mat_x)),
    'feature_mean_variance': float(np.mean(feature_var_diag)),
    'feature_corr_diag_mean': float(np.mean(feature_corr_diag)),
    'feature_corr_offdiag_abs_mean': float(np.mean(np.abs(corr_mat_x[feature_offdiag_mask]))),
    'rkhs_cov_trace': float(np.trace(cov_mat_rkhs)),
    'rkhs_mean_variance': float(np.mean(rkhs_var_diag)),
    'rkhs_corr_diag_mean': float(np.mean(rkhs_corr_diag)),
    'rkhs_corr_offdiag_abs_mean': float(np.mean(np.abs(corr_mat_rkhs[rkhs_offdiag_mask]))),
    'rkhs_centering_row_abs_mean_max': float(np.max(np.abs(np.mean(K_centered, axis=1)))),
}
analysis_summary


### Interpretation Guide

- `feature_covariance_shape = (d, d)` confirms second-order structure in original pixel coordinates.
- `rkhs_covariance_shape = (N, N)` confirms the kernel/RKHS representation in sample coordinates.
- `feature_cov_trace` and `rkhs_cov_trace` measure total variance in each representation (not directly comparable in absolute scale because spaces differ).
- `feature_corr_diag_mean` may be below 1 if some features are nearly constant (e.g., MNIST background pixels), because undefined divisions are stabilized to 0.
- `rkhs_corr_diag_mean` should be near 1 for non-degenerate centered kernel coordinates.
- `rkhs_centering_row_abs_mean_max` near machine precision (e.g., around `1e-16`) indicates correct centering of the Gram matrix.
- Larger `rkhs_corr_offdiag_abs_mean` than in feature space suggests stronger nonlinear dependencies captured by the kernel map.

### Example Numerical Reading (Executed with `N_subset = 600`)

When this notebook was executed with the current defaults, representative values were:

- `feature_cov_trace ≈ 50.9505`, `feature_mean_variance ≈ 0.0650`.
- `feature_corr_diag_mean ≈ 0.7487` (below 1 due to near-constant pixels handled with stabilized zeros).
- `feature_corr_offdiag_abs_mean ≈ 0.0466` (moderate linear cross-pixel dependence on average).
- `rkhs_cov_trace ≈ 0.1213`, `rkhs_mean_variance ≈ 2.02e-4`.
- `rkhs_corr_diag_mean = 1.0`, `rkhs_corr_offdiag_abs_mean ≈ 0.1325`.
- `rkhs_centering_row_abs_mean_max ≈ 2.59e-17`, consistent with numerically correct centering.
